## Validation Script for Porting the Nugraph DA code 

#### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [1]:
%load_ext autoreload
%autoreload 2

#### Append the path of the Nugraph Base conda libraries

In [2]:
import os, sys
sys.path.append('/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages')

#### Import packages for training

In [3]:
from pathlib import Path
import nugraph as ng
import pytorch_lightning as pl
print(ng.__file__)

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/nugraph/nugraph/__init__.py


#### Set the model and data to use

In [4]:
Data = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

#### Declare and configure the data module

In [8]:
nudata = Data(model=Model, batch_size=64, data_source_path='/scratch/7DayLifetime/cerati/concat-final-makeup.tiny.gnn.h5')
nudata.event_classes = ['cc_nue', 'cc_numu', 'cc_nutau', 'nc']
print(nudata)
print(nudata.semantic_classes)
print(nudata.event_classes)

{Train dataloader: size=19868}
{Validation dataloader: size=1103}
{Test dataloader: size=1103}
{Predict dataloader: None}
['MIP', 'HIP', 'shower', 'michel', 'diffuse']
['cc_nue', 'cc_numu', 'cc_nutau', 'nc']


#### Configure network

In [9]:
nugraph = Model(
    in_features=5,
    hit_features=128,
    nexus_features=32,
    instance_features=32,
    interaction_features=32,
    semantic_classes=nudata.semantic_classes, 
    event_classes=nudata.event_classes,
    num_iters=5,
    event_head=True,
    semantic_head=True,
    filter_head=False,
    vertex_head=False,
    instance_head=False,
    use_checkpointing=True,
    lr=0.001)

#### Configure logger and callbacks
Declare a TensorBoard logger and define the output directory, so we can monitor network training. Also, define a callback so we can monitor learning rate evolution.

In [10]:
from pytorch_lightning.loggers import TensorBoardLogger
from datetime import datetime
now = datetime.now()
folder_name = "uBooNE-Tiny-Data-%s" % now.strftime("%H-%M-%S")

In [11]:
os.environ["NUGRAPH_LOG"]='/home/twalton/NuGraphLogs/NuGraphMain/' 
logdir = Path(os.environ["NUGRAPH_LOG"])
logdir.mkdir(parents=True, exist_ok=True)
logger = TensorBoardLogger(save_dir=logdir,name=folder_name) 
callbacks = [
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
    pl.callbacks.ModelCheckpoint(monitor="loss/val", mode="min"),
]

#### Declare trainer and run training
First, we set the training device. To train with a GPU, pass an integer; otherwise, it defaults to CPU training. We then instantiate a PyTorch Lightning trainer and run the training stage, iterating over all batches in the training and validation datasets to optimize model parameters, logging metrics to TensorBoard.

In [ ]:
device_number = 0
accelerator, devices = ng.util.configure_device(device_number)
print(accelerator)
print(devices)

trainer = pl.Trainer(accelerator=accelerator,
                     devices=devices,
                     max_epochs=30,
                     min_epochs=2,
                     logger=logger,
                     callbacks=callbacks)
trainer.fit(nugraph, datamodule=nudata)
trainer.test(datamodule=nudata)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100 80GB PCIe MIG 2g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


gpu
[0]


/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

  | Name             | Type            | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | encoder          | Encoder         | 977    | train | 0    
1 | core_net         | NuGraphCore     | 141 K  | train | 0    
2 | event_decoder    | EventDecoder    | 133    | train | 0    
3 | semantic_decoder | SemanticDecoder | 646    | train | 0    
---------------------------------------------------------------------
142 K     Trainable params
11        Non-trainable params
142 K     Total params
0.572     Total estimated model params size (MB)
85        Modules in train mode
0         Modules in eval mode
0         Total Flops
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: Use

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/t

...

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/t

Validation: |          | 0/? [00:00<?, ?it/s]

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/t

Validation: |          | 0/? [00:00<?, ?it/s]

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y_vtx', 'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/t